## CE13_Tinea_Candidiasis_Classifier

# Step 1: Install the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Step 2: Dataset Paths/Image Processing

In [2]:
train_path = "dataset/train"
test_path = "dataset/test"

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(
    rescale=1./255
)

train_data = train_gen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode="binary"
)

test_data = test_gen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

Found 1063 images belonging to 2 classes.
Found 129 images belonging to 2 classes.


# Step 3: Verification of class

In [3]:
print(train_data.class_indices)
print("Training Images:", train_data.samples)
print("Testing Images:", test_data.samples)

{'Candidiasis': 0, 'Tinea': 1}
Training Images: 1063
Testing Images: 129


# Step 4: Build the model

In [4]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the pretrained layers
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# Step 5: Compiling the model

In [5]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Step 6: Compile

In [11]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Step 7: Callbacks

In [6]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model_check = ModelCheckpoint(
    "models/skin_classifier.keras",
    monitor="val_loss",
    save_best_only=True
)

# Step 8: Train the model

In [7]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=20,
    callbacks=[early_stop, model_check]
)

Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 281s 6s/step - accuracy: 0.7676 - loss: 0.5131 - val_accuracy: 0.8992 - val_loss: 0.3454
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 54s 2s/step - accuracy: 0.8156 - loss: 0.4082 - val_accuracy: 0.8527 - val_loss: 0.3359
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.8467 - loss: 0.3572 - val_accuracy: 0.8682 - val_loss: 0.3383
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 57s 2s/step - accuracy: 0.8674 - loss: 0.3100 - val_accuracy: 0.8915 - val_loss: 0.3250
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 56s 2s/step - accuracy: 0.8645 - loss: 0.3088 - val_accuracy: 0.8760 - val_loss: 0.3181
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 57s 2s/step - accuracy: 0.8589 - loss: 0.3034 - val_accuracy: 0.8837 - val_loss: 0.3059
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.8909 - loss: 0.2723 - val_accuracy: 0.8450 - val_loss: 0.3225
Epoch 8/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.8965 - loss: 0.2519 - val_accuracy: 0.8915 - val_loss

# Step 9: Save the model

In [8]:
model.save("models/skin_classifier.keras")

# Step 10: Evaluate the Model

In [10]:
loss, accuracy = model.evaluate(test_data)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 867ms/step - accuracy: 0.8915 - loss: 0.2818
Test Loss: 0.2817675769329071
Test Accuracy: 0.8914728760719299
